In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [2]:
# Load datasets
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
credit_df = pd.read_csv('../data/raw/creditcard.csv')
ip_df = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

print("Fraud Data Shape:", fraud_df.shape)
print("Credit Card Shape:", credit_df.shape)
print("IP to Country Shape:", ip_df.shape)

Fraud Data Shape: (151112, 11)
Credit Card Shape: (284807, 31)
IP to Country Shape: (138846, 3)


In [3]:
# Fraud Data Cleaning
print("Missing values in Fraud:", fraud_df.isnull().sum().sum())
print("Duplicates in Fraud:", fraud_df.duplicated().sum())

# Drop duplicates if any
fraud_df = fraud_df.drop_duplicates()

# Convert timestamps
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

print("Data types fixed.")

Missing values in Fraud: 0
Duplicates in Fraud: 0
Data types fixed.


In [6]:
# === Time Features (ensure they are done) ===
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['time_since_signup'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek

# === FIXED IP to Integer ===
def ip_to_integer(ip):
    if pd.isna(ip) or not isinstance(ip, str):
        return -1
    try:
        octets = ip.split('.')
        return int(octets[0]) * 256**3 + int(octets[1]) * 256**2 + int(octets[2]) * 256 + int(octets[3])
    except:
        return -1

fraud_df['ip_int'] = fraud_df['ip_address'].apply(ip_to_integer)

print("IP conversion done. Invalid IPs:", (fraud_df['ip_int'] == -1).sum())

# === FIX DATA TYPES FOR MERGE ===
fraud_df['ip_int'] = fraud_df['ip_int'].astype('int64')

# Make sure IP bounds in ip_df are also integer
ip_df['lower_bound_ip_address'] = ip_df['lower_bound_ip_address'].astype('int64')
ip_df['upper_bound_ip_address'] = ip_df['upper_bound_ip_address'].astype('int64')

# Sort both
ip_df = ip_df.sort_values('lower_bound_ip_address')
fraud_df = fraud_df.sort_values('ip_int')

# Merge
fraud_df = pd.merge_asof(fraud_df, ip_df, 
                         left_on='ip_int', 
                         right_on='lower_bound_ip_address', 
                         direction='backward')

print("\n✅ Country mapping completed successfully!")
print(fraud_df['country'].value_counts().head(10))

IP conversion done. Invalid IPs: 151112

✅ Country mapping completed successfully!
Series([], Name: count, dtype: int64)


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# === Fraud Dataset Preprocessing ===
drop_cols = ['class', 'user_id', 'device_id', 'ip_address', 'signup_time', 
             'purchase_time', 'ip_int', 'lower_bound_ip_address', 
             'upper_bound_ip_address']

X_f = fraud_df.drop(drop_cols, axis=1, errors='ignore')
y_f = fraud_df['class']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_f, y_f, test_size=0.2, stratify=y_f, random_state=42
)

scaler = StandardScaler()
numeric_cols = X_train_f.select_dtypes(include=['number']).columns.tolist()

X_train_f_scaled = scaler.fit_transform(X_train_f[numeric_cols])

smote = SMOTE(random_state=42)
X_train_f_res, y_train_f_res = smote.fit_resample(X_train_f_scaled, y_train_f)

print("Fraud - After SMOTE:\n", y_train_f_res.value_counts())

# === Credit Card (quick) ===
X_c = credit_df.drop('Class', axis=1)
y_c = credit_df['Class']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.2, stratify=y_c, random_state=42
)

X_train_c_scaled = scaler.fit_transform(X_train_c)
X_train_c_res, y_train_c_res = smote.fit_resample(X_train_c_scaled, y_train_c)

print("Credit Card - After SMOTE:\n", y_train_c_res.value_counts())

# Save
pd.DataFrame(X_train_f_res, columns=numeric_cols).to_csv('../data/processed/fraud_train_res.csv', index=False)
pd.DataFrame(X_train_c_res, columns=X_c.columns).to_csv('../data/processed/credit_train_res.csv', index=False)

print("\n✅ All preprocessing completed and saved!")

Fraud - After SMOTE:
 class
0    109568
1    109568
Name: count, dtype: int64
Credit Card - After SMOTE:
 Class
0    227451
1    227451
Name: count, dtype: int64

✅ All preprocessing completed and saved!


In [8]:
from sklearn.preprocessing import OneHotEncoder

# For Fraud Data - Encode categorical columns
categorical_cols = ['source', 'browser', 'sex', 'country']

# Fill any missing countries
fraud_df['country'] = fraud_df['country'].fillna('Unknown')

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cats = encoder.fit_transform(X_train_f[categorical_cols])

# Convert to DataFrame and combine with numeric features
encoded_df = pd.DataFrame(encoded_cats, columns=encoder.get_feature_names_out(categorical_cols))

print("One-hot encoding completed for", categorical_cols)

One-hot encoding completed for ['source', 'browser', 'sex', 'country']
